In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# The Zebra Puzzle

The following puzzle appeared in the magazine *Life International* on the 17th of December in the year 1962:
<ol>
    <li>There are five houses.</li>
    <li>The Englishman lives in the red house.</li>
    <li>The Spaniard owns the dog.</li>
    <li>Coffee is drunk in the green house.</li>
    <li>The Ukrainian drinks tea.</li>
    <li>The green house is immediately to the right of the ivory house.</li>
    <li>The Old Gold smoker owns snails.</li>
    <li>Kools are smoked in the yellow house.</li>
    <li>Milk is drunk in the middle house.</li>
    <li>The Norwegian lives in the first house.</li>
    <li>The man who smokes Chesterfields lives in the house next to the man with the fox.</li>
    <li>Kools are smoked in the house next to the house where the horse is kept.</li>
    <li>The Lucky Strike smoker drinks orange juice.</li>
    <li>The Japanese smokes Parliaments.</li>
    <li>The Norwegian lives next to the blue house.</li>
</ol>
Furthermore, each of the five houses is painted in a different colour, their inhabitants are of different nationalities, own different pets, drink different beverages, and smoke different brands of cigarettes.

Your task is to write a program that answers the following questions: 
<ul>
    <li><b>Who drinks water?</b></li>
    <li><b>Who owns the zebra?</b></li>
</ul>

We use one variable for every property, e.g. the variable `English` is the number of the house where the Englishman lives.  First, we have to import the CSP solver.  The notebook `02-Backtracking-Constraint-Solver.ipynb` defines the types `Constraint<'V>` and `CSP<'V>`, the functions `constr`, `different`, `equal`, and `hasValue` that create constraints, and the function `solve`.

In [2]:
importNotebook "02-Backtracking-Constraint-Solver.ipynb"

imported 

02-Backtracking-Constraint-Solver.ipynb

In order to succinctly express the constraints that all houses have different colours, the inhabitants have different nationalities etc., it is convenient to implement a function `allDifferent`.  It takes one argument:
- `Variables` is a list of variables.

It returns a list of constraints that is satisfied if and only if all the variables from `Variables` have different values.

In [3]:
let allDifferent (Variables: string list) : Constraint<'V> list =
    [ for x in Variables do
        for y in Variables do
            if x < y then different x y ]

In [4]:
(allDifferent [ "x"; "y"; "z" ] : Constraint<int> list)

[x ≠ y; x ≠ z; y ≠ z]

The function `nextTo` takes two arguments:
- `x` is a variable,
- `y` is a variable.

It returns a constraint that is satisfied if the houses `x` and `y` are neighbours, i.e. if $|x - y| = 1$.

In [5]:
let nextTo (x: string) (y: string) : Constraint<int> =
    constr (sprintf "|%s - %s| = 1" x y) [x; y] (fun A -> abs (A[x] - A[y]) = 1)

In [6]:
nextTo "Kools" "Horse"

|Kools - Horse| = 1

In [7]:
let Nations = [ "English"; "Spanish"; "Ukrainian"; "Norwegian"; "Japanese" ]
let Drinks  = [ "Coffee"; "Tea"; "Milk"; "OrangeJuice"; "Water" ]
let Pets    = [ "Dog"; "Snails"; "Horse"; "Fox"; "Zebra" ]
let Brands  = [ "LuckyStrike"; "Parliaments"; "Kools"; "Chesterfields"; "OldGold" ]
let Colours = [ "Red"; "Green"; "Ivory"; "Yellow"; "Blue" ]

The function `zebraCSP` takes no arguments.  It returns a CSP that codes the zebra problem.  When implementing this function it is important to order the variables in a way that variables that are connected to each other by a constraint are tried in succession, for otherwise the CSP solver will take mu$\cdots$uch longer.

In [8]:
let zebraCSP () : CSP<int> =
    let mutable Constraints = allDifferent Nations
    Constraints <- Constraints @ allDifferent Drinks
    Constraints <- Constraints @ allDifferent Pets
    Constraints <- Constraints @ allDifferent Brands
    Constraints <- Constraints @ allDifferent Colours
    // The Englishman lives in the red house.
    Constraints <- Constraints @ [ equal "English" "Red" ]
    // The Spaniard owns the dog.
    Constraints <- Constraints @ [ equal "Spanish" "Dog" ]
    // Coffee is drunk in the green house.
    Constraints <- Constraints @ [ equal "Coffee" "Green" ]
    // The Ukrainian drinks tea.
    Constraints <- Constraints @ [ equal "Ukrainian" "Tea" ]
    // The green house is immediately to the right of the ivory house.
    Constraints <- Constraints @ [ constr "Green = Ivory + 1" ["Green"; "Ivory"] (fun A -> A["Green"] = A["Ivory"] + 1) ]
    // The Old Gold smoker owns snails.
    Constraints <- Constraints @ [ equal "OldGold" "Snails" ]
    // Kools are smoked in the yellow house.
    Constraints <- Constraints @ [ equal "Kools" "Yellow" ]
    // Milk is drunk in the middle house.
    Constraints <- Constraints @ [ hasValue "Milk" 3 ]
    // The Norwegian lives in the first house.
    Constraints <- Constraints @ [ hasValue "Norwegian" 1 ]
    // The man who smokes Chesterfields lives in the house next to the man with the fox.
    Constraints <- Constraints @ [ nextTo "Chesterfields" "Fox" ]
    // Kools are smoked in the house next to the house where the horse is kept.
    Constraints <- Constraints @ [ nextTo "Kools" "Horse" ]
    // The Lucky Strike smoker drinks orange juice.
    Constraints <- Constraints @ [ equal "LuckyStrike" "OrangeJuice" ]
    // The Japanese smokes Parliaments.
    Constraints <- Constraints @ [ equal "Japanese" "Parliaments" ]
    // The Norwegian lives next to the blue house.
    Constraints <- Constraints @ [ nextTo "Norwegian" "Blue" ]
    let Variables = [ "Milk"; "Norwegian"; "Blue"; "English"; "Red"; "Spanish"; "Dog"; "Coffee"; "Green"; "Ukrainian"
                      "Tea"; "Ivory"; "OldGold"; "Snails"; "Kools"; "Horse"; "Yellow"; "Japanese"; "Parliaments"
                      "Chesterfields"; "Fox"; "OrangeJuice"; "LuckyStrike"; "Water"; "Zebra" ]
    // There are five houses
    { Variables = Variables; Values = [ 1 .. 5 ]; Constraints = Constraints }

In [9]:
let zebra = zebraCSP ()

In [10]:
zebra

{ Variables = ["Milk"; "Norwegian"; "Blue"; "English"; "Red"; "Spanish"; "Dog"; "Coffee"; "Green"; "Ukrainian"; "Tea"; "Ivory"; "OldGold"; "Snails"; "Kools"; "Horse"; "Yellow"; "Japanese"; "Parliaments"; "Chesterfields"; "Fox"; "OrangeJuice"; "LuckyStrike"; "Water"; "Zebra"]; Values = [1; 2; 3; 4; 5]; Constraints = [English ≠ Spanish; English ≠ Ukrainian; English ≠ Norwegian; English ≠ Japanese; Spanish ≠ Ukrainian; Norwegian ≠ Spanish; Norwegian ≠ Ukrainian; Japanese ≠ Spanish; Japanese ≠ Ukrainian; Japanese ≠ Norwegian; Coffee ≠ Tea; Coffee ≠ Milk; Coffee ≠ OrangeJuice; Coffee ≠ Water; Tea ≠ Water; Milk ≠ Tea; Milk ≠ OrangeJuice; Milk ≠ Water; OrangeJuice ≠ Tea; OrangeJuice ≠ Water; Dog ≠ Snails; Dog ≠ Horse; Dog ≠ Fox; Dog ≠ Zebra; Snails ≠ Zebra; Horse ≠ Snails; Horse ≠ Zebra; Fox ≠ Snails; Fox ≠ Horse; Fox ≠ Zebra; LuckyStrike ≠ Parliaments; LuckyStrike ≠ OldGold; Kools ≠ LuckyStrike; Kools ≠ Parliaments; Kools ≠ OldGold; Chesterfields ≠ LuckyStrike; Chesterfields ≠ Parliaments; C

When the variables are ordered in a sensible way, the problem can be solved in less than a second.  If the variables are ordered randomly, you can expect your computation to take several minutes.

In [11]:
#!time
let solution = (solve zebra).Value

Wall time: 6.6455ms

In [12]:
solution

map [("Blue", 2); ("Chesterfields", 2); ("Coffee", 5); ("Dog", 4); ("English", 3); ("Fox", 1); ("Green", 5); ("Horse", 2); ("Ivory", 4); ("Japanese", 5); ("Kools", 1); ("LuckyStrike", 4); ("Milk", 3); ("Norwegian", 1); ("OldGold", 3); ("OrangeJuice", 4); ("Parliaments", 5); ("Red", 3); ("Snails", 3); ("Spanish", 4); ("Tea", 2); ("Ukrainian", 2); ("Water", 1); ("Yellow", 1); ("Zebra", 5)]

## Functions to Print the Solution

The function `showHTML` takes one argument:
- `Solution` is a map assigning house numbers to all properties.

It returns an HTML table that shows the solution.  The table has one row for every house.  The function `HTML` is provided by the notebook kernel.

In [13]:
let showHTML (Solution: Map<string, int>) =
    let sb = System.Text.StringBuilder()
    sb.Append("<table style=\"border:2px solid blue\">\n<tr>") |> ignore
    for name in [ "House"; "Nationality"; "Drink"; "Animal"; "Brand"; "Colour" ] do
        sb.Append("<th style=\"color:gold; background-color:blue\">" + name + "</th>") |> ignore
    sb.Append("</tr>\n") |> ignore
    for house in 1 .. 5 do
        sb.Append("<tr><td style=\"border:1px solid green\">" + string house + "</td>") |> ignore
        for Class in [ Nations; Drinks; Pets; Brands; Colours ] do
            for x in Class do
                if Solution[x] = house then
                    sb.Append("<td style=\"border:1px solid green\">" + x + "</td>") |> ignore
        sb.Append("</tr>\n") |> ignore
    sb.Append("</table>") |> ignore
    HTML(sb.ToString())

In [14]:
showHTML solution

House,Nationality,Drink,Animal,Brand,Colour
1,Norwegian,Water,Fox,Kools,Yellow
2,Ukrainian,Tea,Horse,Chesterfields,Blue
3,English,Milk,Snails,OldGold,Red
4,Spanish,OrangeJuice,Dog,LuckyStrike,Ivory
5,Japanese,Coffee,Zebra,Parliaments,Green


## Checking the Uniqueness

The function `negateSolution` takes one argument:
- `Solution` is a map that represents a solution.

It returns a constraint that is satisfied iff one of the variables has a value that is different from its value in `Solution`.  This constraint contains all variables.  Hence it is only checked when all variables have a value.

In [15]:
let negateSolution (Solution: Map<string, int>) : Constraint<int> =
    let vars = [ for KeyValue(x, _) in Solution -> x ]
    let text = String.concat " ∨ " [ for KeyValue(x, v) in Solution -> sprintf "%s ≠ %d" x v ]
    constr text vars (fun A -> vars |> List.exists (fun x -> A[x] <> Solution[x]))

In [16]:
negateSolution solution

Blue ≠ 2 ∨ Chesterfields ≠ 2 ∨ Coffee ≠ 5 ∨ Dog ≠ 4 ∨ English ≠ 3 ∨ Fox ≠ 1 ∨ Green ≠ 5 ∨ Horse ≠ 2 ∨ Ivory ≠ 4 ∨ Japanese ≠ 5 ∨ Kools ≠ 1 ∨ LuckyStrike ≠ 4 ∨ Milk ≠ 3 ∨ Norwegian ≠ 1 ∨ OldGold ≠ 3 ∨ OrangeJuice ≠ 4 ∨ Parliaments ≠ 5 ∨ Red ≠ 3 ∨ Snails ≠ 3 ∨ Spanish ≠ 4 ∨ Tea ≠ 2 ∨ Ukrainian ≠ 2 ∨ Water ≠ 1 ∨ Yellow ≠ 1 ∨ Zebra ≠ 5

The function `checkUniqueness` takes two arguments:
* `Solution` is a solution of `csp`.
* `csp` is a *constraint satisfaction problem*.

It tries to compute a new solution for the constraint satisfaction problem that is different from the given solution.  If there is such a solution, it is displayed.

In [17]:
let checkUniqueness (Solution: Map<string, int>) (csp: CSP<int>) =
    let NewCSP = { csp with Constraints = csp.Constraints @ [ negateSolution Solution ] }
    match solve NewCSP with
    | Some other ->
        printfn "The solution is not unique. The alternative solution is:"
        display (showHTML other) |> ignore
    | None ->
        printfn "Well done! The solution is unique."

In [18]:
#!time
checkUniqueness solution zebra

Well done! The solution is unique.

Wall time: 14.9183ms